# 🗂️ The Analyst's Notebook · Part 2
### Session 2

This is the same investigation you started last session. You are a junior analyst building a risk report on a small set of US stocks, and the report grows more detailed as your tools do.

Part 1 ended with a complaint about its own answer. You measured risk as the gap between a stock's highest and lowest close, which uses **two** days out of 252 and throws the rest away. This session you replace it with **volatility**, the standard deviation of every daily return, and you build the calculation yourself out of loops and functions.

You then apply it to five stocks at once and rank them, which is where dictionaries earn their place.

## How to work through this

This is one continuous investigation, not a set of separate exercises. Work **top to bottom, in order**: the functions you write early on are called by the questions that follow, and the numbers you store are reused in the final report.

- Run the **quick load** cell first. It puts last session's data back in memory and restates what you found in Part 1.
- Replace each `...` with real code and run the cell.
- Folded **💡 Hint** and **✅ Solution** blocks sit under each question. Try first, then check.
- One cell in the middle is **broken on purpose** (marked ⚠️) so you can practise reading a real error. Everything else runs cleanly.

**Formulas you will use today**

**Daily return** from one close to the next:

$$r_i=\dfrac{p_i-p_{i-1}}{p_{i-1}}$$

**Average** of a list of returns:

$$\bar{r}=\dfrac{1}{n}\sum_i r_i$$

**Volatility**, the standard deviation of the daily returns:

$$\sigma=\sqrt{\ \dfrac{1}{n}\sum_i \left(r_i-\bar{r}\right)^2\ }$$

In words: take each return's distance from the average, square it, average those squares, and take the square root.

---

## 🔄 Quick load · pick up where Part 1 left off

Run this cell first. It does two jobs:

1. Loads the 2024 daily closes for **five** stocks into a dictionary called `closes`, and also hands you `aapl_closes` and `ko_closes` directly, as in Part 1.
2. Restates the four findings you saved in Part 1, so you can compare your new measure against the old one.

You are not expected to follow the loading code. It borrows pandas, the data-table library introduced in Session 3. For now, just run it.

In [ ]:
# --- Part 1 quick load (provided; you will understand it after Session 3) ---
import os
import pandas as pd

CANDIDATE_DIRS = ["data", os.path.join("..", "data"), "."]

# If the files are not next to the notebook, as in Google Colab, they are
# downloaded from the course repository instead. Nothing to set up.
REPO_RAW_URL = "https://raw.githubusercontent.com/theill95/mlfin-2026/main/data/"

TICKERS = ['AAPL', 'KO', 'NVDA', 'JNJ', 'JPM']


def load_closes(filename):
    """Return the close column of a two-column CSV as a plain list."""
    for folder in CANDIDATE_DIRS:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return pd.read_csv(path)["close"].tolist()
    if REPO_RAW_URL is not None:
        return pd.read_csv(REPO_RAW_URL + filename)["close"].tolist()
    raise FileNotFoundError(
        f"Could not find {filename}. Run this notebook from the course folder "
        f"so the data/ folder is nearby, upload the CSV into Colab, or set "
        f"REPO_RAW_URL to the course repository data URL."
    )


# Every stock's 2024 closes, stored under its ticker
closes = {}
for ticker in TICKERS:
    closes[ticker] = load_closes(ticker.lower() + "_2024_closes.csv")

aapl_closes = closes["AAPL"]
ko_closes = closes["KO"]

# --- What you found in Part 1, restated so you can build on it ---
aapl_return = 0.3556    # Apple's 2024 return
ko_return   = 0.0726    # Coca-Cola's 2024 return
aapl_risk   = 0.5122    # Apple's relative range, Part 1's crude risk gauge
ko_risk     = 0.2615    # Coca-Cola's relative range

print("Loaded", len(closes), "stocks:", TICKERS)
print("Trading days each:", len(aapl_closes))
print(f"Part 1 recap  Apple:     return {aapl_return:.2%}, crude risk {aapl_risk:.2%}")
print(f"Part 1 recap  Coca-Cola: return {ko_return:.2%}, crude risk {ko_risk:.2%}")

---

### 🔁 Q1 · Every daily return of the year

Part 1 computed a single return, from the first close of the year to the last. A proper risk
measure needs **every** day's return.

Fill the empty list with Apple's daily returns, one per pair of consecutive closes:

$$r_i=\dfrac{p_i-p_{i-1}}{p_{i-1}}$$

Start the loop at position **1**, because day 0 has no day before it.

In [ ]:
aapl_returns = []
for i in range(1, len(aapl_closes)):
    ...

print("How many returns:", len(aapl_returns))
print("First three:", aapl_returns[:3])

<details>
<summary>💡 Hint 1</summary>

Inside the loop, today is `aapl_closes[i]` and yesterday is `aapl_closes[i - 1]`.

</details>

<details>
<summary>💡 Hint 2</summary>

Compute `r = (aapl_closes[i] - aapl_closes[i - 1]) / aapl_closes[i - 1]`, then `aapl_returns.append(r)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
aapl_returns = []
for i in range(1, len(aapl_closes)):
    r = (aapl_closes[i] - aapl_closes[i - 1]) / aapl_closes[i - 1]
    aapl_returns.append(r)

print("How many returns:", len(aapl_returns))
print("First three:", aapl_returns[:3])
```

**251 returns** from 252 closes, starting -0.0075, -0.0127, -0.0040. One fewer return than prices, always, because the first day has nothing to be compared against. Doing this by hand in Part 1 would have meant 251 lines.

</details>

---

### 🛠️ Q2 · Make it a function you can reuse

You will need those returns for every stock, not just Apple, so package the loop into a function.

Write `daily_returns(prices)` so it takes any list of prices and hands back the list of returns. The
empty list and the `return` are already in place; write the two lines inside the loop.

In [ ]:
def daily_returns(prices):
    """Daily simple returns from a list of closing prices."""
    result = []
    for i in range(1, len(prices)):
        ...
    return result


# a tiny check you can verify in your head: 100 -> 110 is +10%, 110 -> 99 is -10%
print(daily_returns([100.0, 110.0, 99.0]))

# now the real thing, stored for the questions below
aapl_returns = daily_returns(aapl_closes)
ko_returns = daily_returns(ko_closes)

<details>
<summary>💡 Hint 1</summary>

It is the same two lines as Q1, with `prices` in place of `aapl_closes` and `result` in place of `aapl_returns`.

</details>

<details>
<summary>💡 Hint 2</summary>

`r = (prices[i] - prices[i - 1]) / prices[i - 1]`, then `result.append(r)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
def daily_returns(prices):
    """Daily simple returns from a list of closing prices."""
    result = []
    for i in range(1, len(prices)):
        r = (prices[i] - prices[i - 1]) / prices[i - 1]
        result.append(r)
    return result


print(daily_returns([100.0, 110.0, 99.0]))

aapl_returns = daily_returns(aapl_closes)
ko_returns = daily_returns(ko_closes)
```

`[0.1, -0.1]` on the tiny list, which is the check you wanted: +10% then -10%. The same function now handles Apple's 252 closes and Coca-Cola's without a single change. Testing a function on a small input you can verify by hand, before trusting it on real data, is a habit worth keeping.

</details>

---

### 📐 Q3 · The average daily return

Volatility measures spread **around the average**, so you need the average first.

Write `mean(values)` returning the average of a list, then use it on both stocks.

$$\bar{r}=\dfrac{1}{n}\sum_i r_i$$

In [ ]:
def mean(values):
    """The average of a list of numbers."""
    ...


aapl_mean = mean(aapl_returns)
ko_mean = mean(ko_returns)
print("Apple mean daily return:", aapl_mean)
print("Coca-Cola mean daily return:", ko_mean)

<details>
<summary>💡 Hint 1</summary>

`sum(values) / len(values)` is the whole function body.

</details>

<details>
<summary>💡 Hint 2</summary>

Remember to `return` it, not print it, because the questions below need the number back.

</details>

<details>
<summary>✅ Solution</summary>

```python
def mean(values):
    """The average of a list of numbers."""
    return sum(values) / len(values)


aapl_mean = mean(aapl_returns)
ko_mean = mean(ko_returns)
print("Apple mean daily return:", aapl_mean)
print("Coca-Cola mean daily return:", ko_mean)
```

**Apple 0.1312% a day, Coca-Cola 0.0311% a day.** Both tiny, which is normal: a year's worth of growth arrives in very small daily pieces. Note how little these averages tell you about risk. Apple's average day is barely different from Coca-Cola's, yet the two stocks do not feel remotely alike to hold. The difference is in the spread, which is what you measure next.

</details>

---

> ✅ **Checkpoint** · you have turned a year of prices into a year of returns, and written two functions you will reuse.

---

### 📉 Q4 · Apple's volatility, step by step

Now the spread. Volatility is the square root of the average squared distance from the mean:

$$\sigma=\sqrt{\ \dfrac{1}{n}\sum_i \left(r_i-\bar{r}\right)^2\ }$$

Build it in the open here, one step at a time, using `aapl_returns` and `aapl_mean` from above. Take
the square root with `** 0.5`.

In [ ]:
squared_total = ...
for r in aapl_returns:
    ...

aapl_vol = ...
print("Apple daily volatility:", aapl_vol)

<details>
<summary>💡 Hint 1</summary>

Start `squared_total = 0`. Each pass adds one squared distance: `squared_total += (r - aapl_mean) ** 2`.

</details>

<details>
<summary>💡 Hint 2</summary>

After the loop, divide by how many returns there are and take the square root: `aapl_vol = (squared_total / len(aapl_returns)) ** 0.5`.

</details>

<details>
<summary>✅ Solution</summary>

```python
squared_total = 0
for r in aapl_returns:
    squared_total += (r - aapl_mean) ** 2

aapl_vol = (squared_total / len(aapl_returns)) ** 0.5
print("Apple daily volatility:", aapl_vol)
```

**0.0141, or about 1.41% a day.** That is the typical size of an Apple trading day in 2024. Squaring does two jobs: it removes the sign, so a fall counts as risk exactly like a rise, and it weights large moves far more heavily than small ones. The square root at the end puts the answer back into the units of a daily return, which is what makes it readable.

</details>

---

### 📦 Q5 · Package it, using the functions you already wrote

Doing that by hand for five stocks would be miserable. Package it.

Write `volatility(prices)` that goes all the way from **prices** to a volatility. The two helpers you
wrote above do the first half of the work, so call them rather than writing their loops again. Only
the squared-distance step and the final line are left for you.

In [ ]:
def volatility(prices):
    """Standard deviation of the daily returns of a price series."""
    r = daily_returns(prices)      # your function from Q2
    m = mean(r)                    # your function from Q3
    squared_total = 0
    for x in r:
        ...                        # add this return's squared distance from m
    return ...                     # average the squares, then square-root


ko_vol = volatility(ko_closes)
print("Coca-Cola daily volatility:", ko_vol)
print("Apple again, should match Q4:", volatility(aapl_closes))

<details>
<summary>💡 Hint 1</summary>

The loop line is the same as Q4 with `x` and `m` in place of `r` and `aapl_mean`: `squared_total += (x - m) ** 2`.

</details>

<details>
<summary>💡 Hint 2</summary>

The return line is the same as Q4's last step: `return (squared_total / len(r)) ** 0.5`.

</details>

<details>
<summary>✅ Solution</summary>

```python
def volatility(prices):
    """Standard deviation of the daily returns of a price series."""
    r = daily_returns(prices)
    m = mean(r)
    squared_total = 0
    for x in r:
        squared_total += (x - m) ** 2
    return (squared_total / len(r)) ** 0.5


ko_vol = volatility(ko_closes)
print("Coca-Cola daily volatility:", ko_vol)
print("Apple again, should match Q4:", volatility(aapl_closes))
```

**Coca-Cola 0.0080 (0.80% a day)**, and Apple comes back as 0.0141, matching Q4 exactly. That match is the point of the check: the packaged version and the hand-built one are the same calculation. Notice that `volatility` is short because it delegates. Small functions that call other small functions is how real analysis code is kept readable.

</details>

---

### ⚖️ Q6 · Was Part 1's crude gauge wrong?

You now have two measures of the same thing. Part 1's relative range (`aapl_risk`, `ko_risk`, both
restored by the quick load) used two days out of 252. Your volatility uses all of them.

Build one line that puts them side by side, something like:

`Apple: vol 1.41% vs range 51.22%  |  Coca-Cola: vol 0.80% vs range 26.15%`

Then read the note below, because the interesting part is what the comparison does **not** show.

In [ ]:
comparison = ...
print(comparison)

<details>
<summary>💡 Hint 1</summary>

Build a single f-string holding all four saved values, each formatted with `:.2%`.

</details>

<details>
<summary>💡 Hint 2</summary>

`comparison = f"Apple: vol {aapl_vol:.2%} vs range {aapl_risk:.2%}  |  Coca-Cola: vol {ko_vol:.2%} vs range {ko_risk:.2%}"`

</details>

<details>
<summary>✅ Solution</summary>

```python
comparison = f"Apple: vol {aapl_vol:.2%} vs range {aapl_risk:.2%}  |  Coca-Cola: vol {ko_vol:.2%} vs range {ko_risk:.2%}"
print(comparison)
```

`Apple: vol 1.41% vs range 51.22%  |  Coca-Cola: vol 0.80% vs range 26.15%`.

The two measures disagree wildly in **size** and agree on the **ranking**: Apple is riskier either way. The sizes are not comparable, and should not be: a range is a whole year's travel, a volatility is one typical day. The ratios are worth a look though. By range Apple was 1.96 times Coca-Cola; by volatility it is 1.76 times. So Part 1's gauge was not misleading here, it was just crude, and it overstated the gap. A range is at the mercy of two single days, and one freak session can set it. Volatility is an average over 251 days, so no single day can dominate it. That robustness is why it is the standard measure.

</details>

---

> ✅ **Checkpoint** · you have replaced the crude risk gauge with a proper one, and checked the two against each other.

---

### 🐞 Q7 · Fix a colleague's returns loop

⚠️ **Run the cell below.** A colleague wrote their own returns loop and it crashes. Read the error,
work out which pass fails and why, then write a corrected version underneath.

Your fixed list should be exactly as long as the `aapl_returns` you built earlier.

In [ ]:
broken_returns = []
for i in range(1, len(aapl_closes) + 1):
    broken_returns.append((aapl_closes[i] - aapl_closes[i - 1]) / aapl_closes[i - 1])
print(len(broken_returns))

*Your fix:*

In [ ]:
fixed_returns = []
...

print("How many:", len(fixed_returns))
print("Same as Q1?", len(fixed_returns) == len(aapl_returns))

<details>
<summary>💡 Hint 1</summary>

There are 252 closes, so the valid positions are 0 to 251. The `+ 1` makes the last pass ask for `aapl_closes[252]`, which does not exist.

</details>

<details>
<summary>💡 Hint 2</summary>

Drop the `+ 1`: `for i in range(1, len(aapl_closes)):`. Starting at 1 is correct and must stay, because the body reads `aapl_closes[i - 1]`.

</details>

<details>
<summary>✅ Solution</summary>

```python
fixed_returns = []
for i in range(1, len(aapl_closes)):
    fixed_returns.append((aapl_closes[i] - aapl_closes[i - 1]) / aapl_closes[i - 1])

print("How many:", len(fixed_returns))
print("Same as Q1?", len(fixed_returns) == len(aapl_returns))
```

**251, and it matches Q1.** The bug is an `IndexError` on the very last pass. It is worth being precise about the two ends of this loop, because they are different bugs: the **start** at 1 protects `[i - 1]` at the bottom, and the **stop** at `len(prices)` protects `[i]` at the top. Change either one and you get a wrong answer or a crash.

</details>

---

### 🗂️ Q8 · All five stocks at once

This is where a dictionary earns its keep. `closes` holds five price series, each under its ticker.

Loop over it and build a second dictionary, `vol`, holding each stock's volatility. Your `volatility`
function does all the work, so the loop body is two short lines.

In [ ]:
vol = {}
for ticker in closes:
    ...

print(vol)

<details>
<summary>💡 Hint 1</summary>

Looping a dictionary gives you the **keys**, so `ticker` is `"AAPL"`, `"KO"` and so on.

</details>

<details>
<summary>💡 Hint 2</summary>

Read the prices with `closes[ticker]`, then store the answer with `vol[ticker] = volatility(closes[ticker])`.

</details>

<details>
<summary>✅ Solution</summary>

```python
vol = {}
for ticker in closes:
    vol[ticker] = volatility(closes[ticker])

print(vol)
```

Five volatilities, each under its ticker: **AAPL** 1.41%, **KO** 0.80%, **NVDA** 3.30%, **JNJ** 0.94%, **JPM** 1.48%. One loop applied a function you wrote to every stock you have. Adding a sixth stock would mean adding one CSV, and not one line of this analysis.

</details>

---

### 🏆 Q9 · The riskiest and the calmest

Rank them. Find the ticker with the **highest** volatility and the one with the **lowest**, using
only a loop and an `if`.

Python does have shortcuts for this, and section L of the exercises shows you `sorted` and
`max(vol.values())`. Do it the long way here anyway, once, so you know what those shortcuts are doing
on your behalf.

Keep a best-so-far as you go. Start `highest` at **0**, because every volatility is above zero, and
start `lowest` at **1.0**, because no stock moves 100% on a typical day.

In [ ]:
riskiest = ""
calmest = ""
highest = 0
lowest = 1.0

for ticker in vol:
    ...

print("Riskiest:", riskiest)
print("Calmest: ", calmest)

<details>
<summary>💡 Hint 1</summary>

Two `if` statements inside the one loop, each comparing `vol[ticker]` against the best seen so far.

</details>

<details>
<summary>💡 Hint 2</summary>

`if vol[ticker] > highest:` then update **both** `highest = vol[ticker]` and `riskiest = ticker`. The lowest works the same way with `<` and `lowest`.

</details>

<details>
<summary>✅ Solution</summary>

```python
riskiest = ""
calmest = ""
highest = 0
lowest = 1.0

for ticker in vol:
    if vol[ticker] > highest:
        highest = vol[ticker]
        riskiest = ticker
    if vol[ticker] < lowest:
        lowest = vol[ticker]
        calmest = ticker

print("Riskiest:", riskiest)
print("Calmest: ", calmest)
```

**Riskiest NVDA (3.30% a day), calmest KO (0.80% a day).** Nvidia moves about 4.1 times as much on a typical day as Coca-Cola, which is the sort of gap that decides how much of each you can hold. The pattern here, carry a best-so-far and replace it whenever you find better, is how you find an extreme in one pass, and it is what `max` does internally.

</details>

---

### 📝 Q10 · The page of the report

The payoff. Using the values you have stored, print one line per stock, ordered as they sit in the
dictionary, reading like:

```
AAPL  vol 1.41%
KO    vol 0.80%
```

Then add a closing line naming the riskiest and calmest stock.

In [ ]:
for ticker in vol:
    ...

summary = ...
print(summary)

<details>
<summary>💡 Hint 1</summary>

Inside the loop, `print(ticker, f"vol {vol[ticker]:.2%}")` is enough.

</details>

<details>
<summary>💡 Hint 2</summary>

`summary = f"Riskiest: {riskiest} | Calmest: {calmest}"`, reusing what Q9 stored.

</details>

<details>
<summary>✅ Solution</summary>

```python
for ticker in vol:
    print(ticker, f"vol {vol[ticker]:.2%}")

summary = f"Riskiest: {riskiest} | Calmest: {calmest}"
print(summary)
```

Five formatted lines and a verdict, and every number in them was computed by a function you wrote. `Riskiest: NVDA | Calmest: KO`. This is a real page of a risk report: not a rough gauge from two days, but a measure that used all 251 daily moves of every stock.

</details>

---

> ✅ **Checkpoint** · you have ranked five stocks on a proper measure of risk, built entirely from your own functions.

---

### 🔮 Q11 · Where the case goes next

Part 2 leaves the report in decent shape. You measured risk honestly, on every trading day, for five stocks. Three things are still missing, and each is a later part of the case.

1. **Do it for everything, and see it.** Five stocks took five CSV files and a loop. The full set has eleven, along with a decade of history rather than one year, and none of it has been plotted. A data table (pandas) holds all of it at once, and a chart shows in one glance what a column of numbers hides. *(Session 3.)*
2. **Ask whether it lasts.** You measured Nvidia as the most volatile stock of 2024. Was it also the most volatile in 2023? Does a stock that moved a lot last year tend to move a lot next year? If volatility persists, then measuring it is not only description, it is the beginning of a forecast. *(Session 4.)*
3. **Say what you would predict, and from what.** To go further you have to be precise about what you are predicting (a target) and what you would use to predict it (features), and about never letting information from the future leak into either. That framing is the foundation of machine learning. *(Session 4.)*

No model is fitted in these four sessions. The work here is the honest groundwork: measure carefully, then state exactly what question you would ask next.

---

## 📌 Part 2 findings · carry into Part 3

The investigation now leaves these in the notebook's memory:

- `daily_returns(prices)`, `mean(values)` and `volatility(prices)`: three functions you wrote
- `aapl_vol` and `ko_vol`: the two stocks from Part 1, measured properly
- `vol`: a dictionary of all five stocks' volatilities
- `riskiest` and `calmest`: the two ends of the ranking

Apple's daily volatility is 1.41% against Coca-Cola's 0.80%, so the Part 1 ranking survives a much better measurement. Across all five, NVDA is the most volatile at 3.30% a day and KO the least at 0.80%.

Session 3 continues from here. Everything you did with loops and dictionaries, pandas does on a whole table at once, for eleven stocks and ten years, and matplotlib finally lets you look at it.

## 🏁 Part 2 complete

Using loops, conditions, functions and dictionaries, you turned a year of prices into a year of returns, built the standard measure of risk from its definition, checked it against the crude gauge from Part 1, fixed an off-by-one bug, and ranked five stocks without a sorting function.

The measure you built by hand here is the one every risk system in the industry starts from. From Session 3 you will get it in a single call, and you will know exactly what that call is doing.

*Stuck for more than 15 minutes? Ask a friend, ask an AI for a hint (not the answer), or email me at `jobo@econ.au.dk`.*